# From the coursework to a model factory
### making one training loop **fast on every machine**, for CNNs *and* Transformers

*A CSED 504 **mini-capstone.** Across 501 → 502 → 503 we learned these models by hand. Then we spent
**months** making them actually *train* — fast, correctly, on every machine we could get (**Mac / Windows
/ Linux / Colab**), and at a scale the coursework never went near: **ImageNet-32, 1.28M images.** This
notebook is the receipt. It **trains a real model**, **measures** the tricks (and the traps) that made it
fast, and **predicts** what the next model / next dataset will cost. It's the story of the rewrites — the
bugs we hit dragging a from-scratch coursework loop onto Apple MPS, a Colab T4, a Windows box with two
Blackwell cards, and a Linux server — and why every one of them mattered for the science.*

## The ladder: how much of the training do *you* write?

- **CSED 501 — classical ML in `sklearn` / `pandas`.** Regression, classification, clustering (hand-coded
  **K-means/K-means++**), a first `sklearn` MLP + tabular RL (`a4`); bias/variance, Ridge, SVM kernels,
  probability. Here **`model.fit()` *is* the training** — no loop, no gradient, no device.
- **CSED 502 — drop to the metal.** Backprop, a hand-written **`Solver`** (`csed502/.../solver.py`),
  BatchNorm + ConvNets from scratch, *then* the PyTorch intro (`csed502/assignment4/PyTorch.ipynb`) and
  hand-built attention for captioning (`a5`). **You** write the loop.
- **CSED 503 — industrialize it.** Linear text models → **self-attention from scratch** (`503/a4`,
  minGPT) → using GPT-2/Qwen; the loop goes to HuggingFace's `Trainer`, and the first **cross-platform
  sweep** appears (`csed503/assignment1/hyper_sweep.py` + `cuda_check.py`).

**`sklearn.fit()` → write the loop by hand → industrialize it.** Two rungs were only *just* reached:

1. **GPU training.** First shows up in `502/a4` as a `USE_GPU` flag + `.to(device)` — that's *it*. No AMP,
   no multi-GPU, no data-pipeline tuning. 503 reached `Trainer(fp16=True)` + `DataParallel`, but **we
   never wrote mixed precision ourselves.**
2. **Vision Transformers.** We hand-built attention for *sequences*. A **ViT** — attention over image
   patches — we never touched.

`503`'s `hyper_sweep.py` + `cuda_check.py` were the **factory seed** (the latter is the direct ancestor
of this repo's `src/common/gpu_check.py`). This notebook adds the next rung — and shows the scars.

## 1. From `sklearn.fit()` to a `Solver` to the loop

In **501** you called `model.fit()` and never saw the loop. In **502** you *wrote* it — `Solver._step()`
by hand. PyTorch is that same loop once more, and it's the one piece PyTorch still makes you write — so
it's where every "make it fast" trick below has to live. (`cifar10_train.ipynb` walks the mapping in full.)

| CSED 501 | CSED 502, by hand | PyTorch (here) |
|---|---|---|
| `sklearn` did it | `softmax_loss` (`layers.py`) | `nn.CrossEntropyLoss` |
| `sklearn` did it | `ThreeLayerConvNet` (`cnn.py`) | `torchvision.models.resnet18` |
| `model.fit()` | `Solver._step()` (`solver.py`) | the `for x, y in loader:` loop |
| `sklearn` did it | `Solver._save_checkpoint()` | `torch.save(model.state_dict())` |

In [ ]:
# ../common = gpu_check (the grown-up cuda_check.py); ../a1-imagenet32 = models.py; ../perf = perfkit
import os, sys, time, math
for rel in ('../common', '../a1-imagenet32', '../perf'):
    p = os.path.normpath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from gpu_check import get_device, set_seed, enable_fast_matmul
import models as M
set_seed(42)

## 2. One device, four platforms

502's GPU cell was three lines and broke the moment a teammate opened it on a **Mac** (Apple **MPS**,
not CUDA), on **Colab** (a different GPU every session), or on a **CPU** laptop. `get_device()` is what
`csed503/cuda_check.py` grew into: **CUDA → MPS → CPU**, an MPS smoke-test (some macOS builds advertise
MPS then crash on the first op), and best-same-arch selection on a multi-GPU box. Same notebook, four
platforms.

In [ ]:
DEVICE = get_device()
if DEVICE.type == 'cuda':
    enable_fast_matmul()                 # TF32 + cuDNN autotune
# One AMP dtype choice, made once, for whatever hardware this is:
def pick_amp(device):
    if device.type == 'cuda':
        return (torch.bfloat16, False) if torch.cuda.is_bf16_supported() else (torch.float16, True)  # T4 -> fp16
    if device.type == 'cpu':
        return torch.bfloat16, False     # Zen4 AVX-512-BF16; harmless elsewhere
    return None, False                   # MPS: autocast is patchy -> skip
AMP_DTYPE, USE_SCALER = pick_amp(DEVICE)
print(f'\nDevice {DEVICE} | AMP dtype {AMP_DTYPE} | loss scaler {USE_SCALER}')

## 3. The first wall — *feeding* the GPU

502 handed data to the model with a `DataLoader` and CPU worker processes. That is the right tool for
big images — and it broke us twice at 32×32:

- **Spawn.** On Windows/macOS, DataLoader workers are *spawned* (fresh Python processes) and must
  **re-import** the `Dataset` class. A class defined in a notebook cell lives in `__main__` and can't be
  re-imported — the workers die with `Can't get attribute '...'`. So multi-worker loading was
  Linux/Colab-only, and everyone else fell back to `num_workers=0`: **one CPU core** feeding the GPU.
- **The GPU was faster than one core could feed it.** At 32×32 a ResNet-18 trains faster than CPU
  augmentation produces batches, so the card sat *waiting on the CPU*.

The rewrite: **stop using the CPU for data.** CIFAR-10 is ~180 MB — hold the whole thing **on the GPU**
and crop/flip/normalize there, in batches. No DataLoader, no workers, no host→device copy. Let's measure
the difference on *this* machine.

In [ ]:
# CIFAR-10, once, as device tensors.
from torchvision.datasets import CIFAR10
from gpu_data import GPUImageLoader                 # the on-device loader we built (src/a1-cv/gpu_data.py)
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
_root = os.path.join(os.getcwd(), 'data')
_tr = CIFAR10(_root, train=True, download=True); _te = CIFAR10(_root, train=False, download=True)
def to_device(ds):
    x = torch.from_numpy(ds.data).to(DEVICE).permute(0, 3, 1, 2).contiguous()   # NHWC uint8 -> NCHW
    return x, torch.tensor(ds.targets, device=DEVICE)
xtr, ytr = to_device(_tr); xte, yte = to_device(_te)
print(f'train {tuple(xtr.shape)}  test {tuple(xte.shape)}  — all on {DEVICE}')

In [ ]:
# Measure: a CPU DataLoader (num_workers=0, the fallback everyone hit) vs the on-device loader.
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class _CPUAug(Dataset):                    # a normal torchvision pipeline
    def __init__(self, imgs, labels):
        self.imgs, self.labels = imgs, labels
        self.tf = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                      transforms.RandomHorizontalFlip(),
                                      transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return self.tf(Image.fromarray(self.imgs[i])), int(self.labels[i])

def feed_rate(loader, n_batches=20):       # imgs/sec this loader can hand the model (fwd only)
    model = M.build('resnet18', 10).to(DEVICE).eval(); seen = 0
    it = iter(loader); torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_batches):
            try: x, y = next(it)
            except StopIteration: it = iter(loader); x, y = next(it)
            x = x.to(DEVICE, non_blocking=True); model(x); seen += x.size(0)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    return seen / (time.time() - t0)

cpu_loader = DataLoader(_CPUAug(_tr.data, _tr.targets), batch_size=512, shuffle=True, num_workers=0)
gpu_loader = GPUImageLoader(xtr, ytr, 512, MEAN, STD, train=True, seed=42)
rates = {'CPU DataLoader\n(workers=0)': feed_rate(cpu_loader), 'GPU-resident\n(on-device aug)': feed_rate(gpu_loader)}
for k, v in rates.items(): print(f'{k.splitlines()[0]:20s} {v:9,.0f} img/s')

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.bar(list(rates), list(rates.values()), color=['#c44', '#4a8'])
ax.set_ylabel('images / sec delivered'); ax.set_title('Feeding the GPU at 32×32')
ax.bar_label(ax.containers[0], fmt='%.0f'); plt.tight_layout(); plt.show()

## 4. Where PyTorch fought back — flips and non-contiguous views

Moving augmentation onto the GPU sounds simple. It wasn't. Two rewrites we didn't see coming:

### (a) The horizontal flip that stalled the pipeline
The obvious way to flip a random half of a batch is a boolean-mask write:
```python
x[flip] = x[flip].flip(-1)          # looks fine...
```
But `x[flip]` is *advanced indexing* — under the hood PyTorch calls `nonzero()` on the mask, which is a
**device→host sync**: the GPU must stop and tell the CPU *how many* rows were selected. Two syncs per
batch, every batch, and it silently **breaks CUDA-graph capture**. The fix is branch-free and
bitwise-identical:
```python
x = torch.where(flip.view(-1,1,1,1), x.flip(-1), x)    # no nonzero(), no sync
```
Let's time both.

In [ ]:
def time_flip(fn, iters=400):
    x = torch.randn(512, 3, 32, 32, device=DEVICE)
    flip = torch.rand(512, device=DEVICE) < 0.5
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    t0 = time.time()
    for _ in range(iters): fn(x, flip)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    return iters / (time.time() - t0)

def mask_write(x, flip): x2 = x.clone(); x2[flip] = x2[flip].flip(-1); return x2   # nonzero() -> sync
def where_flip(x, flip): return torch.where(flip.view(-1, 1, 1, 1), x.flip(-1), x)  # sync-free

a, b = time_flip(mask_write), time_flip(where_flip)
print(f'x[flip]=x[flip].flip(-1) : {a:8,.0f} flips/s   (advanced-index -> nonzero -> host sync)')
print(f'torch.where(...)         : {b:8,.0f} flips/s   ({b/a:.1f}x, and CUDA-graph-safe)')

### (b) `.view()` on a non-contiguous tensor — the Mac-only crash
The per-sample random crop gathers pixels with fancy indexing, which returns a **non-contiguous** tensor
(its memory layout no longer matches its shape). On CUDA that's usually tolerated. On **Apple MPS**, the
ViT's backward pass calls `.view()` on exactly such a tensor and **throws** — a "why does this only break
on the Mac?" afternoon. Two lessons, both baked into the code now:

- **`.reshape()` (or an explicit `.contiguous()` first), never bare `.view()`** on anything that's been
  transposed/indexed/permuted — `.view()` *requires* contiguity, `.reshape()` copies if it must.
- "Runs on every platform" is something you **debug into existence**, not a checkbox. The crop returns a
  contiguous result on purpose; the flip is `torch.where`; nothing downstream assumes a layout.

## 5. The counterintuitive one — mixed precision × memory format

Mixed precision (**AMP**) was never in the coursework (503 only got it via `Trainer(fp16=True)`).
`torch.autocast` runs the matmuls in low precision — **but which** low precision, and **which memory
layout**, interact in a way that cost us days.

`channels_last` (NHWC) is cuDNN's preferred conv layout, so "always use it" is the folklore. On this
hardware that folklore is **half wrong**, and the sign flips with the dtype. Measure it yourself:

In [ ]:
def step_rate(amp_dtype, channels_last, iters=60, warmup=15):
    m = M.build('resnet18', 10).to(DEVICE)
    if channels_last: m = m.to(memory_format=torch.channels_last)
    opt = torch.optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
    x = torch.randn(512, 3, 32, 32, device=DEVICE); y = torch.randint(0, 10, (512,), device=DEVICE)
    if channels_last: x = x.contiguous(memory_format=torch.channels_last)
    def step():
        opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            nn.functional.cross_entropy(m(x), y).backward()
        opt.step()
    for _ in range(warmup): step()
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    t0 = time.time()
    for _ in range(iters): step()
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    return 512 * iters / (time.time() - t0)

configs = [('fp32', None, False), ('fp16', torch.float16, False), ('fp16 + chan_last', torch.float16, True),
           ('bf16', torch.bfloat16, False), ('bf16 + chan_last', torch.bfloat16, True)]
rate = {n: step_rate(dt, cl) for n, dt, cl in configs}
base = rate['fp32']
for n in rate: print(f'{n:18s} {rate[n]:9,.0f} img/s   {rate[n]/base:4.2f}x vs fp32')

fig, ax = plt.subplots(figsize=(7, 3.6))
cols = ['#888', '#69c', '#c44', '#48a', '#2a6']
ax.bar(list(rate), list(rate.values()), color=cols)
ax.set_ylabel('img/s (resnet18, bs512)'); ax.set_title('Mixed precision × memory format — same GPU')
ax.bar_label(ax.containers[0], fmt='%.0f'); plt.xticks(rotation=15); plt.tight_layout(); plt.show()

Read the bars. **`channels_last` helps under bf16 and *hurts* under fp16** — on the RTX PRO 6000
we measured bf16+`channels_last` as the fastest config *and* fp16+`channels_last` as ~4× **slower** than
plain fp16 (a pathological cuDNN NHWC-fp16 path). Same flag, opposite sign, decided by the dtype. There
is no substitute for measuring on the target — which is exactly what pushes us toward a **factory** that
measures for us.

The habit underneath all of this: before optimizing, know **what the GPU is waiting on** —

| symptom | you are… | the fix |
|---|---|---|
| low GPU util, one CPU core pegged | **CPU-bound** (data) | more workers, or on-device data (§3) |
| low GPU util, low CPU util | **launch-bound** (batch too small) | bigger batch, CUDA graphs |
| high GPU util | **compute-bound** | you're done — need a bigger model or better card |

## 6. Now train a *real* model — ResNet-18 → low-90s %

Everything above was setup. Here is a full run with the fast path wired in (on-device data, AMP, and
`channels_last` **only if it helped** for our dtype). ~2 min on the workstation GPU, longer on a laptop/
Colab — drop `EPOCHS` if you just want to watch. This is the "high-quality model," and we plot the curves
`Solver` used to hide.

In [ ]:
from tqdm.auto import tqdm
CHAN_LAST = (DEVICE.type == 'cuda' and AMP_DTYPE is torch.bfloat16)     # the measured rule from §5

def train(model, epochs, lr, opt_name='sgd', mixup=False):
    model = model.to(DEVICE)
    if CHAN_LAST: model = model.to(memory_format=torch.channels_last)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt = (torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05) if opt_name == 'adamw'
           else torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=5e-4))
    warm = torch.optim.lr_scheduler.LinearLR(opt, 0.1, total_iters=min(5, epochs - 1) or 1)
    cos = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs - 5))
    sched = torch.optim.lr_scheduler.SequentialLR(opt, [warm, cos], [min(5, epochs - 1) or 1])
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_SCALER)
    hist = {'loss': [], 'top1': [], 'img_s': []}
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); seen = 0; run = 0.0
        for x, y in tqdm(train_loader, desc=f'epoch {ep}/{epochs}', leave=False):
            if CHAN_LAST: x = x.contiguous(memory_format=torch.channels_last)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
                if mixup:
                    lam = float(np.random.beta(0.2, 0.2)); perm = torch.randperm(x.size(0), device=DEVICE)
                    out = model(lam * x + (1 - lam) * x[perm]); loss = lam * crit(out, y) + (1 - lam) * crit(out, y[perm])
                else:
                    loss = crit(model(x), y)
            if USE_SCALER: scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else: loss.backward(); opt.step()
            run += loss.item() * x.size(0); seen += x.size(0)
        sched.step()
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        hist['loss'].append(run / seen); hist['top1'].append(accuracy(model)); hist['img_s'].append(seen / (time.time() - t0))
        print(f'epoch {ep:2d}/{epochs}  loss {run/seen:.3f}  val top1 {hist["top1"][-1]:6.2%}  {hist["img_s"][-1]:8,.0f} img/s')
    return hist

@torch.no_grad()
def accuracy(model):
    model.eval(); c = n = 0
    for x, y in test_loader:
        if CHAN_LAST: x = x.contiguous(memory_format=torch.channels_last)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            c += (model(x).argmax(1) == y).sum().item()
        n += y.size(0)
    return c / n

train_loader = GPUImageLoader(xtr, ytr, 512, MEAN, STD, train=True, seed=42)
test_loader  = GPUImageLoader(xte, yte, 512, MEAN, STD, train=False)
EPOCHS = 24                                    # ~92% on a GPU in ~2 min; lower it to just watch
resnet_hist = train(M.build('resnet18', 10), EPOCHS, lr=0.1 * 512 / 256)
print(f'\nResNet-18 best val top-1: {max(resnet_hist["top1"]):.2%}')

In [ ]:
# The curves Solver used to hide.
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
xs = range(1, len(resnet_hist['top1']) + 1)
a1.plot(xs, resnet_hist['loss'], color='#c44'); a1.set_title('train loss'); a1.set_xlabel('epoch')
a2.plot(xs, resnet_hist['top1'], color='#2a6'); a2.set_title(f"val top-1 (best {max(resnet_hist['top1']):.1%})")
a2.set_xlabel('epoch'); a2.set_ylim(0, 1); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7. The same job, a Transformer — new recipe, same fast loop

A **ViT** is the attention we hand-built in class, now over image *patches*. It ports with one argument
(`num_classes`) — but the **training recipe does not**. Hand it the ResNet's recipe (SGD, `lr=0.1`) and
it lands in the 50s; the fixes are **AdamW**, LR **warmup**, and **mixup**. Same fast loop (on-device
data, AMP, the channels_last rule), just a transformer recipe. On 50k images the CNN still wins — that
gap is the ViT's missing inductive bias, and it's the whole point of the crossover study.

In [ ]:
vit_hist = train(M.build('vit', 10), EPOCHS, lr=1e-3, opt_name='adamw', mixup=True)
print(f'\nViT best val top-1: {max(vit_hist["top1"]):.2%}   '
      f'(ResNet-18 was {max(resnet_hist["top1"]):.2%} — the CNN wins on 50k images)')

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(range(1, len(resnet_hist['top1'])+1), resnet_hist['top1'], label='ResNet-18 (CNN)', color='#2a6')
ax.plot(range(1, len(vit_hist['top1'])+1), vit_hist['top1'], label='ViT (Transformer)', color='#96c')
ax.set_xlabel('epoch'); ax.set_ylabel('val top-1'); ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=.3)
ax.set_title('CNN vs Transformer on CIFAR-10 (50k images)'); plt.tight_layout(); plt.show()

### The payoff of all that speed — the crossover *only shows up at scale*

On **50k** images the CNN wins big (~92% vs ~73% for a well-tuned ViT). Its locality + translation
priors are simply worth more than anything a transformer can learn from so little data. The obvious
conclusion — "CNNs beat ViTs" — is **wrong**, and proving it wrong is the whole project.

Run the *same two models* on **ImageNet-32 — 1.28M images, 25× the data, identical 32×32 size** (they
port with `num_classes` alone) — and the result **flips**: the from-scratch **ViT overtakes**, val top-1
**42.76% vs the ResNet's 41.54%**. Enough data, and the transformer *learns* the priors we hardwired into
the CNN, then keeps going (`../README.md`, `../a1-imagenet32/`).

Here's the capstone point: **you cannot even run that experiment without §3–§5.** 1.28M images at 32×32 is
exactly where a `num_workers=0` DataLoader, an `fp16 + channels_last` misstep, or a per-batch host sync
turns a **27-minute** run into an all-day one — times two models, times every ablation. The months of
optimization aren't a side quest; they're **what makes the science affordable.** And once training is
cheap and predictable, you stop running experiments one at a time and build a *factory* to search them.

## 8. Predict the cost *before* you pay it — dataset size × model

`hyper_sweep.py` (503) searched configs by **running** them. A factory can't: a bad choice can be a
2-day run. `../perf/` adds what the coursework didn't — **estimate wall-clock before training** from the
machine's measured ceilings and the model's FLOPs. Below we probe this GPU once, then predict a 40-epoch
run for **four models × a range of dataset sizes** (CIFAR-10/100 = 50k, ImageNet-32 = 1.28M, and beyond),
and plot the cost surface. This is the factory's redesign gate — *"2 minutes or 2 days?"* — answered in
seconds, no training required.

In [ ]:
import perfkit as pk
peak = pk.probe_matmul_tflops(DEVICE)                    # quick burst GEMM (seconds)
p_tflops = peak.get('bf16') or peak.get('tf32') or peak.get('fp32')
print(f"{pk.fingerprint(DEVICE).get('gpu', DEVICE.type)}: ~{p_tflops:.0f} TFLOPS peak\n")

MODELS = ['resnet18', 'vit', 'resnet50', 'vit_base']
SIZES  = [50_000, 200_000, 1_281_167, 5_000_000]         # CIFAR -> ImageNet-32 -> beyond
flops = {m: pk.count_flops_per_image(lambda m=m: M.build(m, 1000)) for m in MODELS}

def predict_hours(model, n_train, epochs=40):
    work = pk.workload_spec(f'{model}', n_train=n_train, n_val=10_000, batch_size=512, epochs=epochs, flops=flops[model])
    est = pk.tier1_estimate(work, p_tflops=p_tflops)      # instant roofline (optimistic..pessimistic MFU band)
    return est['optimistic']['total_s'] / 3600, est['pessimistic']['total_s'] / 3600

print(f"{'model':10s} " + ' '.join(f'{n//1000:>6d}k' for n in SIZES) + '   (predicted hours, 40 epochs)')
grid = {}
for m in MODELS:
    grid[m] = [predict_hours(m, n) for n in SIZES]
    print(f'{m:10s} ' + ' '.join(f'{lo:6.1f}' for lo, hi in grid[m]))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for m, col in zip(MODELS, ['#2a6', '#96c', '#087', '#639']):
    lo = [h[0] for h in grid[m]]; hi = [h[1] for h in grid[m]]
    ax.plot(SIZES, lo, '-o', color=col, label=m)
    ax.fill_between(SIZES, lo, hi, color=col, alpha=0.12)     # MFU-band uncertainty
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('training-set size (images)'); ax.set_ylabel('predicted wall-clock (hours, 40 epochs)')
ax.set_title(f'Cost vs scale on {pk.fingerprint(DEVICE).get("gpu", DEVICE.type)}')
for n, tag in [(50_000, 'CIFAR'), (1_281_167, 'ImageNet-32')]:
    ax.axvline(n, ls=':', color='#bbb'); ax.text(n, ax.get_ylim()[0], tag, rotation=90, va='bottom', fontsize=8, color='#888')
ax.legend(); ax.grid(alpha=.3, which='both'); plt.tight_layout(); plt.show()
print('Shaded band = the honest unknown (achievable GPU efficiency); the line is the optimistic edge.')

## 9. The model factory

Stack the pieces and you have the backbone `hyper_sweep.py` was reaching for:

- **runs anywhere** — one `get_device()`, four platforms (§2), debugged through the MPS/spawn/`.view()`
  scars (§3–4);
- **fast** — on-device data, AMP, the measured `channels_last`-by-dtype rule, the bound-diagnosis habit (§3–5);
- **both families** — CNN *and* Transformer, each with its own recipe, on the same loop (§6–7);
- **predictable** — cost estimated before the run, so **search replaces babysitting** (§8).

A **model factory** points that at a search space — architecture family, width/depth, recipe — and lets
it *estimate → schedule → run* across whatever hardware is on hand (the two RTX PRO 6000s here, a Mac
overnight, a Colab tab), keeping the accuracy/cost Pareto front. The **what to search** is the
CNN-vs-Transformer-vs-data-scale question (`../README.md`, `../a1-imagenet32/`, `cifar100_hf_train.ipynb`);
the **how to run it cheaply, everywhere** is everything above.

We started at `sklearn.fit()` and a hand-written `Solver`. This is where that goes.